# STREAM-1D Solver Verification & Visualisation

This notebook demonstrates and verifies the **STREAM-1D** native Python bindings. By compiling the identical Rust core used in the WebAssembly module, the Python module runs high-fidelity hydraulic computations (open channel flow, composite roughness, and structure losses) with zero execution overhead.

Here, we will:
1. Run a steady-state profile verification on the HEC-RAS ConSpan test project (Q = 1000 cfs, featuring an arch culvert).
2. Plot the resulting water surface profile (WSEL) and critical depths.
3. Run an integrated bridge pier loss simulation demonstrating Yarnell head loss calculations.
4. Tabulate results in a Pandas DataFrame.

In [ ]:
# Install dependencies if needed (e.g. on Binder)
# %pip install pandas matplotlib maturin

import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import streams1d as st
print("STREAM-1D package imported successfully!")

## 1. Load HEC-RAS ConSpan Verification Project Data

We load the ConSpan verification geometry and extract the culvert parameters.

In [ ]:
# Path to the ConSpan verification dataset
dataset_path = os.path.join("verification", "conspan_project_12.json")
with open(dataset_path, "r") as f:
    project = json.load(f)

# Extract geometries and map to streams1d.CrossSection objects
xs_raw = project["geometry_data"]
cross_sections = []
for xs in xs_raw:
    cross_sections.append(
        st.CrossSection(
            station=float(xs["station"]),
            x=[float(val) for val in xs["x"]],
            y=[float(val) for val in xs["y"]],
            n_stations=[float(val) for val in xs["n_stations"]],
            n_values=[float(val) for val in xs["n_values"]],
            unit_system=xs.get("unit_system", "USCustomary"),
            is_overbank=xs.get("is_overbank")
        )
    )

# Extract culvert data
culverts = project.get("culvert_stations", [])
culvert_stations = [float(c["station"]) for c in culverts]
culvert_shape_types = [int(c["shape_type"]) for c in culverts]
culvert_spans = [float(c["span"]) for c in culverts]
culvert_rises = [float(c["rise"]) for c in culverts]
culvert_roughness_ns = [float(c["roughness_n"]) for c in culverts]
culvert_lengths = [float(c["length"]) for c in culverts]
culvert_entrance_loss_coeffs = [float(c["entrance_loss_coeff"]) for c in culverts]
culvert_exit_loss_coeffs = [float(c["exit_loss_coeff"]) for c in culverts]
culvert_barrels = [int(c.get("num_barrels", 1)) for c in culverts]
culvert_roughness_n_bottoms = [float(c.get("roughness_n_bottom", c["roughness_n"])) for c in culverts]
culvert_depth_bottom_ns = [float(c.get("depth_bottom_n", 0.0)) for c in culverts]
culvert_depth_blockeds = [float(c.get("depth_blocked", 0.0)) for c in culverts]

print(f"Loaded {len(cross_sections)} cross-sections and {len(culvert_stations)} culvert structures.")

## 2. Solve Steady-State Profile (Q = 1000 cfs)

We instantiate [SteadyInputs](file:///) and execute the standard-step backwater calculation.

In [ ]:
inputs = st.SteadyInputs(
    cross_sections=cross_sections,
    flow_rate=1000.0,
    num_slices=int(project["parameters"].get("vertical_slices", 100)),
    coeff_contraction=0.1,
    coeff_expansion=0.3,
    regime=int(project["parameters"].get("flow_regime", 0)),
    downstream_wsel=30.51,
    max_spacing=float(project["parameters"].get("max_spacing", 100.0)),
    # Boundary conditions
    downstream_bc_type=0, # Known WSEL
    downstream_bc_slope=float(project["parameters"].get("downstream_bc_slope", 0.001)),
    downstream_bc_rating_q=[],
    downstream_bc_rating_wsel=[],
    upstream_bc_type=0,
    upstream_bc_slope=float(project["parameters"].get("upstream_bc_slope", 0.01)),
    upstream_bc_rating_q=[],
    upstream_bc_rating_wsel=[],
    upstream_wsel=float(project["parameters"].get("upstream_wsel", 15.0)),
    # Culverts
    culvert_stations=culvert_stations,
    culvert_shape_types=culvert_shape_types,
    culvert_spans=culvert_spans,
    culvert_rises=culvert_rises,
    culvert_roughness_ns=culvert_roughness_ns,
    culvert_lengths=culvert_lengths,
    culvert_entrance_loss_coeffs=culvert_entrance_loss_coeffs,
    culvert_exit_loss_coeffs=culvert_exit_loss_coeffs,
    culvert_barrels=culvert_barrels,
    culvert_roughness_n_bottoms=culvert_roughness_n_bottoms,
    culvert_depth_bottom_ns=culvert_depth_bottom_ns,
    culvert_depth_blockeds=culvert_depth_blockeds,
)

results = st.solve_steady(inputs)
print("Steady sweep completed successfully.")

## 3. Compare with HEC-RAS Expected Results

Let's compare STREAM-1D calculations against HEC-RAS.

In [ ]:
# Verification target elevations
expected_wsel = {
    2827.0: 33.720,
    1257.0: 32.920,
    0.0: 30.510
}

station_list = [float(xs["station"]) for xs in xs_raw]
table_data = []
for station, expected_val in expected_wsel.items():
    if station in station_list:
        idx = station_list.index(station)
        calc_val = results["wsel"][idx]
        diff = calc_val - expected_val
        table_data.append({
            "Station": int(station),
            "Calculated WSEL (ft)": round(calc_val, 3),
            "HEC-RAS WSEL (ft)": round(expected_val, 3),
            "Difference (ft)": round(diff, 3),
            "Status": "PASS" if abs(diff) <= 0.04 else "FAIL"
        })

df = pd.DataFrame(table_data)
print(df.to_string(index=False))

## 4. Plot Water Surface Profile

We plot the channel bed invert, the computed WSEL, and critical depth elevations along the reach.

In [ ]:
stations = [float(xs.station) for xs in cross_sections]
bed_inverts = [min(xs.y) for xs in cross_sections]
wsel = results["wsel"]
crit_wsel = results["critical_wsel"]

plt.figure(figsize=(10, 6))
plt.plot(stations, bed_inverts, label="Bed Invert", color="black", linewidth=2)
plt.plot(stations, wsel, label="Water Surface (WSEL)", color="dodgerblue", linewidth=2.5)
plt.plot(stations, crit_wsel, label="Critical WSEL", color="red", linestyle="--", linewidth=1.5)

# Highlight culvert zone
plt.axvspan(1200, 1257, color="gray", alpha=0.3, label="Culvert Obstruction Zone")

plt.title("STREAM-1D Water Surface Profile - HEC-RAS ConSpan Verification", fontsize=14)
plt.xlabel("Stationing (ft)", fontsize=12)
plt.ylabel("Elevation (ft)", fontsize=12)
plt.legend(loc="upper right", fontsize=10)
plt.grid(True, linestyle=":", alpha=0.6)
plt.gca().invert_xaxis()  # Downstream is 0 (right), upstream is positive (left)
plt.show()

## 5. Verify Bridge & Pier Hydraulics

This simulation validates the **Yarnell Pier head loss equation** on a 3-section reach with a bridge deck and square pier obstruction.

In [ ]:
# Setup a 200m reach with a bridge deck at station 50
xs200 = st.CrossSection(200.0, [0.0, 0.0, 10.0, 10.0], [10.2, 0.2, 0.2, 10.2], [0.0], [0.03], "Metric")
xs100 = st.CrossSection(100.0, [0.0, 0.0, 10.0, 10.0], [10.1, 0.1, 0.1, 10.1], [0.0], [0.03], "Metric")
xs0 = st.CrossSection(0.0, [0.0, 0.0, 10.0, 10.0], [10.0, 0.0, 0.0, 10.0], [0.0], [0.03], "Metric")

bridge_inputs = st.SteadyInputs(
    cross_sections=[xs200, xs100, xs0],
    flow_rate=15.0,  # 15 cms
    num_slices=50,
    regime=0,        # Subcritical
    downstream_wsel=3.0,
    downstream_bc_type=0,
    bridge_stations=[50.0],
    bridge_low_chords=[5.0],
    bridge_high_chords=[7.0],
    bridge_pier_widths=[0.5], # 0.5m pier width
    bridge_num_piers=[2],
    bridge_pier_shapes=[0],
    bridge_weir_coeffs=[1.44],
    bridge_orifice_coeffs=[0.5]
)

bridge_results = st.solve_steady(bridge_inputs)
ds_wsel = bridge_results["wsel"][2]
us_wsel = bridge_results["wsel"][1]
backwater = us_wsel - ds_wsel

print(f"Bridge Downstream WSEL (Sta 0):   {ds_wsel:.4f} m")
print(f"Bridge Upstream WSEL (Sta 100):   {us_wsel:.4f} m")
print(f"Pier Headwater Backwater Loss:    {backwater:.4f} m")